In [ ]:
import sys

sys.path.append("/mnt/jinho/Development/Projects/2026/BMW_LLMs_finetuning/src")

import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

from utils import retrieve_config

CONFIG_PATH = "/mnt/jinho/Development/Projects/2026/BMW_LLMs_finetuning/config/config.yaml"
CONFIG_DATA = retrieve_config(CONFIG_PATH, "data")
CONFIG_LLMS = retrieve_config(CONFIG_PATH, "llms")
CONFIG_TRAIN = retrieve_config(CONFIG_PATH, "train")
CONFIG_GEN = retrieve_config(CONFIG_PATH, "generation")

set_seed(CONFIG_TRAIN.get("seed", 42))

In [ ]:
MODEL_NAME = CONFIG_LLMS.get("model", "gpt2-large")
DATA_PATH = f"{CONFIG_DATA.get('db_root', 'database')}/{CONFIG_DATA.get('prepare',{}).get('processed_data_fname', 'articles_proc.jsonl')}"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

CONFIG_GEN["eos_token_id"] = tokenizer.eos_token_id

In [10]:
root_full_ft = "/mnt/jinho/Development/Projects/2026/BMW_LLMs_finetuning/results/20260224_124734"
root_m1b_ft = "/mnt/jinho/Development/Projects/2026/BMW_LLMs_finetuning/results/20260224_124927"

base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto").to(DEVICE)
base.config.pad_token_id = tokenizer.eos_token_id
base.eval()

ft_full = AutoPeftModelForCausalLM.from_pretrained(root_full_ft, torch_dtype="auto").to(DEVICE)
ft_full.config.pad_token_id = tokenizer.eos_token_id
ft_full.eval()

ft_m1b = AutoPeftModelForCausalLM.from_pretrained(root_m1b_ft, torch_dtype="auto").to(DEVICE)
ft_m1b.config.pad_token_id = tokenizer.eos_token_id
ft_m1b.eval()

/mnt/jinho/Development/conda/envs/bmw_llms/lib/python3.10/site-packages/peft/peft_model.py:598: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.transformer.h.35.attn.c_attn.lora_A.default.weight', 'base_model.model.transformer.h.35.attn.c_attn.lora_B.default.weight'].
  warnings.warn(warn_message)


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 1280)
        (wpe): Embedding(1024, 1280)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-35): 36 x GPT2Block(
            (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2SdpaAttention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D()
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1280, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=3840, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (

In [ ]:
def generate(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, **CONFIG_GEN)
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [12]:
prompts = [
    "Q: What is BMW Welt?\nA:",
    "Write a BMW press-release style paragraph about BMW Welt in 2026.",
    "Summarize the key highlights of BMW Welt in 2026 in 3 bullet points.",
]


for p in prompts:
    print("=" * 80)
    print("\n--- BASE ---")
    print(generate(base, p))
    print()

    print("\n--- FT (Full) ---")
    print(generate(ft_full, p))
    print()

    print("\n--- FT (LoRA) ---")
    print(generate(ft_m1b, p))
    print()


--- BASE ---


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Q: What is BMW Welt?
A: It's the German word for "work". It's a brand, and it's not just a name. The people who make it are dedicated to making a good car, and that means you can expect quality. We want to create an experience where everyone is satisfied, so our customers need to feel at home. In other words, we want to do things right.

Q: How many employees does BMW Welt employ?

A: About 30-35. There are a lot of people involved in design, development, marketing, sales, finance, logistics, engineering, and so


--- FT (Full) ---


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Q: What is BMW Welt?
A: BMW Welt is a daily calendar for the year. It's available in German, English and French.
B: BMW Welt is a calendar that features all kinds of events. The calendar consists of more than 100 events on everything from sports to culture. The event calendar includes: BMW Welt Live, an annual festival with concerts, exhibitions and events in Munich and other cities; BMW Welt Festival, which attracts thousands of visitors each year; BMW Welt Tour, a free event organized by the Bavarian Car Club in Munich; and BMW Welt Meet, which brings together people and cars


--- FT (LoRA) ---


...

Write a BMW press-release style paragraph about BMW Welt in 2026.
The first paragraph of your BMW press-release should be:
"The new BMW Welt will set the pace for all of our BMW brand, and is an ambitious project that has been put together by our creative leadership team."
The second paragraph of your BMW press-release should be:
"The new BMW Welt is a high-performance vehicle designed to take on the demands of the 21st century. It will be produced in Germany at the BMW Group plant in Mechelen, with its production line situated in the Bavarian heartland. The BMW Welt is a


--- BASE ---


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Summarize the key highlights of BMW Welt in 2026 in 3 bullet points.

• The new BMW 1 Series will be available in seven color combinations, including two distinct colors in gold and silver:

• In the 1 Series Coupe, it will be available in a range of colors:

• In the 1 Series M5, it will be available in a range of colors:

• In the 1 Series X5, it will be available in a range of colors:

• In the 1 Series 2 Series, it will be available in a range of colors:

• In the 1 Series Gran Turismo, it will be available


--- FT (Full) ---


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Summarize the key highlights of BMW Welt in 2026 in 3 bullet points.
1. BMW Welt will be fully integrated into the BMW Group.
2. The BMW Group is now a true "world class" company with global headquarters and operations in more than 200 countries.
3. In addition to the BMW Group, the BMW Group also has subsidiaries in China and Japan.
4. The BMW Group will continue to build on its strengths in all markets: the BMW Group will continue to produce innovative vehicles and provide the best customer service in the world.
5. The BMW Group continues to offer a full range of high-quality, reliable and technologically advanced products


--- FT (LoRA) ---
Summarize the key highlights of BMW Welt in 2026 in 3 bullet points.
<Titles>: BMW Welt: In 2026, BMW Welt will feature an extensive range of products from leading brands in the premium mobility sector. The BMW Welt brand and its brand management will be fully integrated into BMW Group. BMW Welt: A new model and new opportunities for the entire 

# Evaulate Q&A performance of the fine-tuned model


User BERTScore for sementic similiarity

EM (Exact match) doesn't work.

F1 is also not a good choice. Not classification task.


In [13]:
import json
import re
from collections import Counter

import pandas as pd

QNA_PATH = "/mnt/jinho/Development/Projects/2026/BMW_LLMs_finetuning/database/QnA.jsonl"


def load_qna(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


def extract_answer(full_text):
    if "A:" in full_text:
        return full_text.split("A:", 1)[1].strip().split("\n")[0]
    return full_text.strip().split("\n")[0]


def generate_answer(model, question):
    prompt = f"Q: {question}\nA:"
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    eval_gen_cfg = dict(CONFIG_GEN)
    eval_gen_cfg["do_sample"] = False
    eval_gen_cfg["max_new_tokens"] = min(eval_gen_cfg.get("max_new_tokens", 64), 32)
    eval_gen_cfg["pad_token_id"] = tokenizer.eos_token_id
    eval_gen_cfg["eos_token_id"] = tokenizer.eos_token_id
    eval_gen_cfg.pop("temperature", None)
    eval_gen_cfg.pop("top_p", None)

    with torch.no_grad():
        output_ids = model.generate(**inputs, **eval_gen_cfg)

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return extract_answer(decoded)


qna_rows = load_qna(QNA_PATH)
print(f"Loaded {len(qna_rows)} Q&A samples from {QNA_PATH}")

Loaded 20 Q&A samples from /mnt/jinho/Development/Projects/2026/BMW_LLMs_finetuning/database/QnA.jsonl


In [ ]:
from bert_score import score as bert_score


def compute_bertscore(df, model_name, scorer_model="distilbert-base-uncased"):
    cands = df["prediction"].tolist()
    refs = df["reference"].tolist()

    p, r, f1 = bert_score(
        cands,
        refs,
        lang="en",
        model_type=scorer_model,
        device=DEVICE,
        verbose=False,
    )

    return {
        "model": model_name,
        "bert_p": float(p.mean().item()),
        "bert_r": float(r.mean().item()),
        "bert_f1": float(f1.mean().item()),
    }

In [15]:
import numpy as np

models = {
    "base": base,
    "ft_full": ft_full,
    "ft_m1b": ft_m1b,
}

all_results = []

for model_name, model_obj in models.items():
    result_per_model = []
    for row in qna_rows:
        pred = generate_answer(model_obj, row["question"])
        ref = row["answer"]

        p, r, f1 = bert_score(
            [pred],
            [ref],
            lang="en",
            model_type="distilbert-base-uncased",
            device=DEVICE,
            verbose=False,
        )

        result_per_model.append(
            {
                "model": model_name,
                "id": row["id"],
                "question": row["question"],
                "reference": ref,
                "prediction": pred,
                "bert_p": float(p.mean().item()),
                "bert_r": float(r.mean().item()),
                "bert_f1": float(f1.mean().item()),
            }
        )

    all_results.extend(result_per_model)

/mnt/jinho/Development/conda/envs/bmw_llms/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [17]:
all_results_df = pd.DataFrame(all_results)
summary = (
    all_results_df.groupby("model", as_index=False)
    .agg(bert_p=("bert_p", "mean"), bert_r=("bert_r", "mean"), bert_f1=("bert_f1", "mean"))
    .sort_values("bert_p")
)

summary.round(4)

,model,bert_p,bert_r,bert_f1
0,base,0.7042,0.7647,0.7325
2,ft_m1b,0.7085,0.7585,0.7323
1,ft_full,0.7110,0.7569,0.7328
